# Denoising Diffusion Probababilistic Models

## Forward Process

$q(x_t|x_{t-1}) \sim \mathcal{N}(x_t; \sqrt{1-\beta_t}x_t, \beta_tI) \\ $
$x_t = \sqrt{\bar{\alpha}_t}x_0 + \sqrt{1-\bar{\alpha}_t}\epsilon$ where $\alpha_t = 1 - \beta_t$ and $\bar{\alpha}_t = \prod \limits_{i=1}^{t}\alpha_i$

## Backward Process
$\bar{x}_0 = \frac{x_t - \sqrt{1-\bar{\alpha}_t}\epsilon_\theta(x_t)}{\sqrt{\bar{\alpha}_t}} \\ $ 
$\mu_\theta = \frac{x_t}{\sqrt{\alpha_t}} - \frac{(1-\alpha_t)(\sqrt{1-\bar{\alpha}_t})}{(1-\bar{\alpha_t})\sqrt{\alpha_t}}\epsilon_\theta \\$
$\Sigma^2 = \frac{(1-\alpha_t)(1-\bar{\alpha}_{t-1})}{1-\bar{\alpha}_t}$

In [1]:
import torch
from einops import rearrange

class LinearNoiseScheduler:
    def __init__(self, num_timesteps, beta_start, beta_end):
        self.num_timesteps = num_timesteps
        self.beta_start = beta_start
        self.beta_end = beta_end

        self.betas = torch.linspace(beta_start, beta_end, num_timesteps)
        self.alphas = 1. - self.betas
        self.alpha_bar = torch.cumprod(self.alphas, dim=0)
        self.sqrt_alpha_bar = torch.sqrt(self.alpha_bar)
        self.sqrt_one_min_alpha_bar = torch.sqrt(1. - self.alpha_bar)

    def forward_process(self, x_0, noise, t):
        original_shape = x_0.shape #BxCxHxW
        batch_size = x_0.shape[0]
        
        assert t.shape[0] == batch_size
        sqrt_alpha_bar = rearrange(self.sqrt_alpha_bar.to(x_0.device)[t], 't -> t 1 1 1')
        sqrt_one_min_alpha_bar = rearrange(self.sqrt_one_min_alpha_bar.to(x_0.device)[t], 't -> t 1 1 1')
        
        return sqrt_alpha_bar * x_0 + sqrt_one_min_alpha_bar*noise
    
    def reverse_process(self, x_t, noise_pred, t):
        #print("debug", x_t.shape, noise_pred.shape, t.shape)
        x_0_bar = (x_t - self.sqrt_one_min_alpha_bar.to(x_t.device)[t]*noise_pred) / self.sqrt_alpha_bar.to(x_t.device)[t]
        x_0_bar = torch.clamp(x_0_bar, -1, 1)

        mu_theta = (x_t - (self.betas.to(x_t.device)[t]*noise_pred/self.sqrt_one_min_alpha_bar.to(x_t.device)[t]))/torch.sqrt(self.alphas.to(x_t.device)[t])

        if t == 0:
            return mu_theta, x_0_bar
        else:
            sigma = torch.sqrt(self.betas.to(x_t.device)[t]*(1 - self.alpha_bar.to(x_t.device)[t-1])/(1- self.alpha_bar.to(x_t.device)[t]))
            z = torch.randn_like(x_t).to(x_t.device)
            return mu_theta + sigma*z, x_0_bar

# Denoiser
## TimeEmbedding Block
$TimeEmb: \mathbb{R}^B \rightarrow \mathbb{R}^{B \times t\_emb\_dim} \\$
$TimeEmb: Linear \circ  SiLU \circ Linear \circ Pos. Enc$

## UNet 
> DownBlock(x) + UpBlock(MidBlock(Downblock(x))) ## resnet
### DownBlock
1. Resnet +  Self.Attention
> CNN: Conv(SiLu(Norm(x)))
> 
> Resnet: x + CNN(CNN(x) + Linear(FC(SiLU(TimeEmb(t)))))
> 
> Attention: Resnet(x) +  SA(Norm(Resnet(x)))
> 
> Downsample

In [2]:
def get_time_embedding(time_steps, t_emb_dim):
    factor = 1e4 ** ((
        torch.arange(
            start = 0,
            end = t_emb_dim // 2,
            device = time_steps.device,
            ) / (t_emb_dim // 2)
        )
    )
    t_emb = time_steps[:, None].repeat(1, t_emb_dim // 2) / factor 
    t_emb = torch.cat([torch.sin(t_emb), torch.cos(t_emb)], dim = -1)
    return t_emb

In [3]:
# ! pip install mamba-ssm

In [4]:
from torch import nn

class DownBlock(nn.Module):
    def __init__(self, in_channels, out_channels, t_emb_dim, down_sample, num_heads):
        super().__init__()
        self.down_sample = down_sample
        self.resnet_conv_first = nn.Sequential(
            nn.GroupNorm(8, in_channels),
            nn.SiLU(),
            nn.Conv2d(in_channels, out_channels, 3, 1, 1)
        )
        self.t_emb_layers = nn.Sequential(
            nn.SiLU(),
            nn.Linear(t_emb_dim, out_channels)
        )
        self.resnet_conv_second = nn.Sequential(
            nn.GroupNorm(8, out_channels),
            nn.SiLU(),
            nn.Conv2d(out_channels, out_channels, 3, 1, 1)
        )

        self.attention_norm = nn.GroupNorm(8, out_channels)
        self.attention = nn.MultiheadAttention(out_channels, num_heads, batch_first=True)

        self.residual_input_conv =  nn.Conv2d(in_channels, out_channels, 1)
        self.down_sample = nn.Conv2d(out_channels, out_channels, 4, 2, 1) if down_sample else nn.Identity()

    def forward(self, x, t_emb):
        out = x
        # Renet Block
        resnet_input = out
        out = self.resnet_conv_first(out)
        out = out + self.t_emb_layers(t_emb)[:, :, None, None] #TXBX1X1
        out = self.resnet_conv_second(out)
        out = out + self.residual_input_conv(resnet_input)

        # Attention Block
        B, C, H, W = out.shape
        in_attn = rearrange(out, 'b c h w -> b c (h w)')
        in_attn = self.attention_norm(in_attn)
        in_attn = rearrange(in_attn, 'b c (h w) -> b (h w) c', h=H, w=W)
        out_attn, _ = self.attention(in_attn, in_attn, in_attn)
        out_attn = rearrange(out_attn, 'b (h w) c -> b c h w', h=H, w=W)
        out = out + out_attn

        ## loop above if multiple layer
        out = self.down_sample(out)
        return out

In [5]:
class MidBlock(nn.Module):
    def __init__(self, in_channels, out_channels, t_emb_dim, num_heads):
        super().__init__()
        self.resnet_conv_first = nn.ModuleList([
            nn.Sequential(
                nn.GroupNorm(8, in_channels),
                nn.SiLU(),
                nn.Conv2d(in_channels, out_channels, 3, 1, 1)
            ),
            nn.Sequential(
                nn.GroupNorm(8, out_channels),
                nn.SiLU(),
                nn.Conv2d(out_channels, out_channels, 3, 1, 1)
            )
        ])

        self.t_emb_layers = nn.ModuleList([
            nn.Sequential(
                nn.SiLU(),
                nn.Linear(t_emb_dim, out_channels)
            ),
            nn.Sequential(
                nn.SiLU(),
                nn.Linear(t_emb_dim, out_channels)
            )
        ])

        self.resnet_conv_second = nn.ModuleList([
            nn.Sequential(
                nn.GroupNorm(8, out_channels),
                nn.SiLU(),
                nn.Conv2d(out_channels, out_channels, 3, 1, 1)
            ),
            nn.Sequential(
                nn.GroupNorm(8, out_channels),
                nn.SiLU(),
                nn.Conv2d(out_channels, out_channels, 3, 1, 1)
            )
        ])

        self.attention_norm = nn.GroupNorm(8, out_channels)
        self.attention = nn.MultiheadAttention(out_channels, num_heads, batch_first=True)
        self.residual_input_conv = nn.ModuleList([
            nn.Conv2d(in_channels, out_channels, 1),
            nn.Conv2d(out_channels, out_channels, 1)
        ])

    def forward(self, x, t_emb):
        out = x
        resnet_input = out

        out = self.resnet_conv_first[0](out)
        out = out + self.t_emb_layers[0](t_emb)[:, :, None, None]
        out = self.resnet_conv_second[0](out)
        out = out + self.residual_input_conv[0](resnet_input)

        B, C, H, W = out.shape
        in_attn = rearrange(out, 'b c h w -> b c (h w)')
        in_attn = self.attention_norm(in_attn)
        in_attn = rearrange(in_attn, 'b c (h w) -> b (h w) c', h=H, w=W)
        out_attn, _ = self.attention(in_attn, in_attn, in_attn)
        out_attn = rearrange(out_attn, 'b (h w) c -> b c h w', h=H, w=W)
        out = out + out_attn

        resnet_input = out
        out = self.resnet_conv_first[1](out)
        out = out + self.t_emb_layers[1](t_emb)[:, :, None, None]
        out = self.resnet_conv_second[1](out)
        out = out + self.residual_input_conv[1](resnet_input)

        return out

In [6]:
class UpBlock(nn.Module):
    def __init__(self, in_channels, out_channels, t_emb_dim, up_sample, num_heads):
        super().__init__()
        self.up_sample = up_sample
        self.resnet_conv_first = nn.Sequential(
            nn.GroupNorm(8, in_channels),
            nn.SiLU(),
            nn.Conv2d(in_channels, out_channels, 3, 1, 1)
        )
        self.t_emb_layers = nn.Sequential(
            nn.SiLU(),
            nn.Linear(t_emb_dim, out_channels)
        )
        self.resnet_conv_second = nn.Sequential(
            nn.GroupNorm(8, out_channels),
            nn.SiLU(),
            nn.Conv2d(out_channels, out_channels, 3, 1, 1)
        )

        self.attention_norm = nn.GroupNorm(8, out_channels)
        self.attention = nn.MultiheadAttention(out_channels, num_heads, batch_first=True)

        self.residual_input_conv = nn.Conv2d(in_channels, out_channels, 1)
        self.up_sample = nn.ConvTranspose2d(in_channels//2, in_channels//2, 4, 2, 1) if up_sample else nn.Identity()

    def forward(self, x, out_down, t_emb):
        x = self.up_sample(x)
        x = torch.cat([x, out_down], dim=1)

        # Renet Block
        # Loop below if multiple layers
        out = x
        resnet_input = out
        out = self.resnet_conv_first(out)
        out = out + self.t_emb_layers(t_emb)[:, :, None, None]
        out = self.resnet_conv_second(out)
        out = out + self.residual_input_conv(resnet_input)

        # Attention Block
        B, C, H, W = out.shape
        in_attn = rearrange(out, 'b c h w -> b c (h w)')
        in_attn = self.attention_norm(in_attn)
        in_attn = rearrange(in_attn, 'b c (h w) -> b (h w) c', h=H, w=W)
        out_attn, _ = self.attention(in_attn, in_attn, in_attn)
        out_attn = rearrange(out_attn, 'b (h w) c -> b c h w', h=H, w=W)
        out = out + out_attn

        return out

In [7]:
class Unet(nn.Module):
    def __init__(self, im_channels, **kwargs):
        super().__init__()
        self.im_channels = im_channels
        self.down_channels = [32, 64, 128, 256]
        self.mid_channels = [256, 256, 128]
        self.t_emb_dim = 128
        self.down_sample = [True, True, False]

        self.t_proj = nn.Sequential(
            nn.Linear(self.t_emb_dim, self.t_emb_dim),
            nn.SiLU(),
            nn.Linear(self.t_emb_dim, self.t_emb_dim)
        )

        self.up_sample = list(reversed(self.down_sample))
        self.conv_in = nn.Conv2d(im_channels, self.down_channels[0], 3, 1, 1)

        self.downs = nn.ModuleList()
        for i in range(len(self.down_channels) - 1):
            self.downs.append(
                DownBlock(self.down_channels[i], self.down_channels[i+1], self.t_emb_dim, self.down_sample[i],  num_heads=4)
            )
        
        self.mids = nn.ModuleList()
        for i in range(len(self.mid_channels) - 1):
            self.mids.append(
                MidBlock(self.mid_channels[i], self.mid_channels[i + 1], self.t_emb_dim, num_heads=4)
            )

        self.ups = nn.ModuleList()
        for i in reversed(range(len(self.down_channels) - 1)):
            self.ups.append(
                UpBlock(self.down_channels[i]*2, self.down_channels[i-1] if i!=0 else 16, self.t_emb_dim, self.down_sample[i], num_heads=4)
            )

        self.norm_out = nn.GroupNorm(8, 16)
        self.conv_out = nn.Conv2d(16, im_channels, 3, 1, 1)

    def forward(self, x, t):
        assert x.shape[1] == self.im_channels
        out = self.conv_in(x)
        t_emb = self.t_proj(get_time_embedding(t, self.t_emb_dim))

        down_outs = []
        #print("down")
        for down in self.downs:
            #print(out.shape)
            down_outs.append(out)
            out = down(out, t_emb)

        #print("mid")
        for mid in self.mids:
            #print(out.shape)
            out = mid(out, t_emb)

        #print("up")
        for up in self.ups:
            down_out = down_outs.pop()
            #print(out.shape, down_out.shape)
            out = up(out, down_out, t_emb)

        out = self.norm_out(out)
        out = nn.SiLU()(out)
        out = self.conv_out(out)

        return out

In [8]:
unet = Unet(im_channels=1)
x = torch.randn(4, 1, 28, 28)
# random time steps in one dimension
t = torch.randint(0, 100, (4,))
t.shape
unet(x, t).shape

torch.Size([4, 1, 28, 28])

In [9]:
# compute the number of parameters
sum(p.numel() for p in unet.parameters())

6067857

# Mamba 

In [10]:
! pip install causal-conv1d>=1.4.0

In [11]:
# Create a mamba_ssm model
from mamba_ssm import Mamba2

/home/gpagluba/anaconda3/envs/env-3.12/lib/python3.12/site-packages/mamba_ssm/ops/selective_scan_interface.py:163: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd
/home/gpagluba/anaconda3/envs/env-3.12/lib/python3.12/site-packages/mamba_ssm/ops/selective_scan_interface.py:239: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd
/home/gpagluba/anaconda3/envs/env-3.12/lib/python3.12/site-packages/mamba_ssm/ops/triton/layer_norm.py:985: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd
/home/gpagluba/anaconda3/envs/env-3.12/lib/python3.12/site-packages/mamba_ssm/ops/triton/layer_norm.py:1044: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custo

In [14]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# warm up
m = Mamba2(
                d_model=49, # Model dimension d_model
                headdim=49, # Head dimension
                expand=8,
            )  
m.to(device)
m(torch.randn(4, 16, 49).to(device)).shape

In [24]:
class S6Unet(nn.Module):
    def __init__(self, im_channels, **kwargs):
        super().__init__()
        self.im_channels = im_channels
        self.down_channels = [32, 64, 128, 256]
        self.mid_channels = [256, 256, 128]
        self.t_emb_dim = 128
        self.down_sample = [True, True, False]

        self.t_proj = nn.Sequential(
            nn.Linear(self.t_emb_dim, self.t_emb_dim),
            nn.SiLU(),
            nn.Linear(self.t_emb_dim, self.t_emb_dim)
        )

        self.up_sample = list(reversed(self.down_sample))
        self.conv_in = nn.Conv2d(im_channels, self.down_channels[0], 3, 1, 1)

        self.downs = nn.ModuleList()
        for i in range(len(self.down_channels) - 1):
            self.downs.append(
                DownBlock(self.down_channels[i], self.down_channels[i+1], self.t_emb_dim, self.down_sample[i],  num_heads=4)
            )
        
        self.mamba_blocks = nn.ModuleList([
            Mamba2(
                d_model=49, # Model dimension d_model
                headdim=49, # Head dimension
                expand=8,
            )   
            for _ in range(2)
        ])
            

        # self.mids = nn.ModuleList()
        # for i in range(len(self.mid_channels) - 1):
        #     self.mids.append(
        #         MidBlock(self.mid_channels[i], self.mid_channels[i + 1], self.t_emb_dim, num_heads=4)
        #     )

        # self.ups = nn.ModuleList()
        # for i in reversed(range(len(self.down_channels) - 1)):
        #     self.ups.append(
        #         UpBlock(self.down_channels[i]*2, self.down_channels[i-1] if i!=0 else 16, self.t_emb_dim, self.down_sample[i], num_heads=4)
        #     )

        self.norm_out = nn.GroupNorm(8, 16)
        self.conv_out = nn.Conv2d(16, im_channels, 3, 1, 1)

    def forward(self, x, t):
        assert x.shape[1] == self.im_channels
        out = self.conv_in(x)
        t_emb = self.t_proj(get_time_embedding(t, self.t_emb_dim))

        down_outs = []
        #print("down")
        for down in self.downs:
            #print(out.shape)
            down_outs.append(out)
            out = down(out, t_emb)
        
        out = rearrange(out, 'b c h w -> b c (h w)')

        print(out.shape)
        print(out.device)
        for mamba_block in self.mamba_blocks:
            out = mamba_block(out)

        return out

In [25]:
device

device(type='cuda')

In [34]:
s6_unet = S6Unet(im_channels=1).to(device)
x = torch.randn(4, 1, 28, 28).float().to(device)
# random time steps in one dimension
t = torch.randint(0, 100, (4,)).to(device)
t.shape
s6_unet(x, t).shape

torch.Size([4, 256, 49])
cuda:0


torch.Size([4, 256, 49])

In [33]:
# compute the number of parameters
sum(p.numel() for p in s6_unet.parameters())

2121281

In [8]:
import glob
import os

import torchvision
from PIL import Image
from tqdm import tqdm
from torch.utils.data.dataloader import DataLoader
from torch.utils.data.dataset import Dataset


class MnistDataset(Dataset):
    r"""
    Nothing special here. Just a simple dataset class for mnist images.
    Created a dataset class rather using torchvision to allow
    replacement with any other image dataset
    """
    def __init__(self, split, im_path, im_ext='png'):
        r"""
        Init method for initializing the dataset properties
        :param split: train/test to locate the image files
        :param im_path: root folder of images
        :param im_ext: image extension. assumes all
        images would be this type.
        """
        self.split = split
        self.im_ext = im_ext
        self.images, self.labels = self.load_images(im_path)
    
    def load_images(self, im_path):
        r"""
        Gets all images from the path specified
        and stacks them all up
        :param im_path:
        :return:
        """
        assert os.path.exists(im_path), "images path {} does not exist".format(im_path)
        ims = []
        labels = []
        for d_name in tqdm(os.listdir(im_path)):
            for fname in glob.glob(os.path.join(im_path, d_name, '*.{}'.format(self.im_ext))):
                ims.append(fname)
                labels.append(int(d_name))
        print('Found {} images for split {}'.format(len(ims), self.split))
        return ims, labels
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, index):
        im = Image.open(self.images[index])
        im_tensor = torchvision.transforms.ToTensor()(im)
        
        # Convert input to -1 to 1 range.
        im_tensor = (2 * im_tensor) - 1
        return im_tensor

# Data Preparation

In [9]:
#! pip install opencv-python

In [114]:
import os
import cv2
from tqdm import tqdm
import numpy as np
import _csv as csv

def extract_images(save_dir, csv_fname):
    assert os.path.exists(save_dir), "Directory {} to save images does not exist".format(save_dir)
    assert os.path.exists(csv_fname), "Csv file {} does not exist".format(csv_fname)
    with open(csv_fname) as f:
        reader = csv.reader(f)
        for idx, row in enumerate(reader):
            if idx == 0:
                continue
            im = np.zeros((784))
            im[:] = list(map(int, row[1:]))
            im = im.reshape((28,28))
            if not os.path.exists(os.path.join(save_dir, row[0])):
                os.mkdir(os.path.join(save_dir, row[0]))
            cv2.imwrite(os.path.join(save_dir, row[0], '{}.png'.format(idx)), im)
            if idx % 1000 == 0:
                print('Finished creating {} images in {}'.format(idx+1, save_dir))
            
            
if __name__ == '__main__':
    CD = '/mnt/c/Users/Acer/Graduate/Courses/AI 222/mlops-learning/exporations/diffusion'
    extract_images(f"{CD}/data/train/images", f"{CD}/data/mnist_train.csv")
    extract_images(f"{CD}/data/test/images", f"{CD}/data/mnist_test.csv")

Finished creating 1001 images in /mnt/c/Users/Acer/Graduate/Courses/AI 222/mlops-learning/exporations/diffusion/data/train/images
Finished creating 2001 images in /mnt/c/Users/Acer/Graduate/Courses/AI 222/mlops-learning/exporations/diffusion/data/train/images
Finished creating 3001 images in /mnt/c/Users/Acer/Graduate/Courses/AI 222/mlops-learning/exporations/diffusion/data/train/images
Finished creating 4001 images in /mnt/c/Users/Acer/Graduate/Courses/AI 222/mlops-learning/exporations/diffusion/data/train/images
Finished creating 5001 images in /mnt/c/Users/Acer/Graduate/Courses/AI 222/mlops-learning/exporations/diffusion/data/train/images
Finished creating 6001 images in /mnt/c/Users/Acer/Graduate/Courses/AI 222/mlops-learning/exporations/diffusion/data/train/images
Finished creating 7001 images in /mnt/c/Users/Acer/Graduate/Courses/AI 222/mlops-learning/exporations/diffusion/data/train/images
Finished creating 8001 images in /mnt/c/Users/Acer/Graduate/Courses/AI 222/mlops-learning/

# Training

In [21]:
import torch
import yaml
import argparse
import os
import numpy as np
from tqdm import tqdm
from torch.optim import Adam
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def train(args):
    # Read the config file #
    with open(args['config_path'], 'r') as file:
        try:
            config = yaml.safe_load(file)
        except yaml.YAMLError as exc:
            print(exc)
    print(config)
    ########################
    
    diffusion_config = config['diffusion_params']
    dataset_config = config['dataset_params']
    model_config = config['model_params']
    train_config = config['train_params']
    
    print(model_config)
    # Create the noise scheduler
    scheduler = LinearNoiseScheduler(num_timesteps=diffusion_config['num_timesteps'],
                                     beta_start=diffusion_config['beta_start'],
                                     beta_end=diffusion_config['beta_end'])
    
    # Create the dataset
    mnist = MnistDataset('train', im_path=dataset_config['im_path'])
    mnist_loader = DataLoader(mnist, batch_size=train_config['batch_size'], shuffle=True, num_workers=4)
    
    # Instantiate the model
    model = Unet(**model_config).to(device)
    model.train()
    
    # Create output directories
    if not os.path.exists(train_config['task_name']):
        os.mkdir(train_config['task_name'])
    
    # Load checkpoint if found
    if os.path.exists(os.path.join(train_config['task_name'],train_config['ckpt_name'])):
        print('Loading checkpoint as found one')
        model.load_state_dict(torch.load(os.path.join(train_config['task_name'],
                                                      train_config['ckpt_name']), map_location=device))
    # Specify training parameters
    num_epochs = train_config['num_epochs']
    optimizer = Adam(model.parameters(), lr=train_config['lr'])
    criterion = torch.nn.MSELoss()
    
    # Run training
    for epoch_idx in range(num_epochs):
        losses = []
        for im in tqdm(mnist_loader):
            optimizer.zero_grad()
            im = im.float().to(device)
            
            # Sample random noise
            noise = torch.randn_like(im).to(device)
            
            # Sample timestep
            t = torch.randint(0, diffusion_config['num_timesteps'], (im.shape[0],)).to(device)
            
            # Add noise to images according to timestep
            noisy_im = scheduler.forward_process(im, noise, t)
            noise_pred = model(noisy_im, t)

            loss = criterion(noise_pred, noise)
            losses.append(loss.item())
            loss.backward()
            optimizer.step()
        print('Finished epoch:{} | Loss : {:.4f}'.format(
            epoch_idx + 1,
            np.mean(losses),
        ))
        torch.save(model.state_dict(), os.path.join(train_config['task_name'],
                                                    train_config['ckpt_name']))
    
    print('Done Training ...')
    

if __name__ == '__main__':
    # parser = argparse.ArgumentParser(description='Arguments for ddpm training')
    import os
    print(os.getcwd())
    train({'config_path': '/mnt/c/Users/Acer/Graduate/Courses/AI 222/mlops-learning/explorations
    
    /diffusion/config/default.yaml'})

/mnt/c/Users/Acer/Graduate/Courses/AI 222/mlops-learning
{'dataset_params': {'im_path': '/mnt/c/Users/Acer/Graduate/Courses/AI 222/mlops-learning/exporations/diffusion/data/train/images'}, 'diffusion_params': {'num_timesteps': 1000, 'beta_start': 0.0001, 'beta_end': 0.02}, 'model_params': {'im_channels': 1, 'im_size': 28, 'down_channels': [32, 64, 128, 256], 'mid_channels': [256, 256, 128], 'down_sample': [True, True, False], 'time_emb_dim': 128, 'num_down_layers': 2, 'num_mid_layers': 2, 'num_up_layers': 2, 'num_heads': 4}, 'train_params': {'task_name': 'default', 'batch_size': 64, 'num_epochs': 40, 'num_samples': 100, 'num_grid_rows': 10, 'lr': 0.0001, 'ckpt_name': '/mnt/c/Users/Acer/Graduate/Courses/AI 222/mlops-learning/exporations/diffusion/ddpm_ckpt.pth'}}
{'im_channels': 1, 'im_size': 28, 'down_channels': [32, 64, 128, 256], 'mid_channels': [256, 256, 128], 'down_sample': [True, True, False], 'time_emb_dim': 128, 'num_down_layers': 2, 'num_mid_layers': 2, 'num_up_layers': 2, 'nu

  0%|                                                                                                                              | 0/10 [00:00<?, ?it/s]

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 37.86it/s]


Found 60000 images for split train


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 938/938 [03:14<00:00,  4.83it/s]


...

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 938/938 [03:14<00:00,  4.81it/s]


Finished epoch:12 | Loss : 0.0250


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 938/938 [03:15<00:00,  4.81it/s]


Finished epoch:13 | Loss : 0.0248


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████▊| 936/938 [03:13<00:00,  4.83it/s]

# Test

In [13]:
import torch
import torchvision
import argparse
import yaml
import os
from torchvision.utils import make_grid
from tqdm import tqdm


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def sample(model, scheduler, train_config, model_config, diffusion_config):
    r"""
    Sample stepwise by going backward one timestep at a time.
    We save the x0 predictions
    """
    xt = torch.randn((train_config['num_samples'],
                      model_config['im_channels'],
                      model_config['im_size'],
                      model_config['im_size'])).to(device)
    for i in tqdm(reversed(range(diffusion_config['num_timesteps']))):
        # Get prediction of noise
        noise_pred = model(xt, torch.as_tensor(i).unsqueeze(0).to(device))
        
        # Use scheduler to get x0 and xt-1
        xt, x0_pred = scheduler.reverse_process(xt, noise_pred, torch.as_tensor(i).to(device))
        
        # Save x0
        ims = torch.clamp(xt, -1., 1.).detach().cpu()
        ims = (ims + 1) / 2
        grid = make_grid(ims, nrow=train_config['num_grid_rows'])
        img = torchvision.transforms.ToPILImage()(grid)
        if not os.path.exists(os.path.join(train_config['task_name'], 'samples')):
            os.mkdir(os.path.join(train_config['task_name'], 'samples'))
        img.save(os.path.join(train_config['task_name'], 'samples', 'x0_{}.png'.format(i)))
        img.close()


def infer(args):
    # Read the config file #
    with open(args['config_path'], 'r') as file:
        try:
            config = yaml.safe_load(file)
        except yaml.YAMLError as exc:
            print(exc)
    print(config)
    ########################
    
    diffusion_config = config['diffusion_params']
    model_config = config['model_params']
    train_config = config['train_params']
    
    # Load model with checkpoint
    model = Unet(**model_config).to(device)
    model.load_state_dict(torch.load(os.path.join(train_config['task_name'],
                                                  train_config['ckpt_name']), map_location=device))
    model.eval()
    
    # Create the noise scheduler
    scheduler = LinearNoiseScheduler(num_timesteps=diffusion_config['num_timesteps'],
                                     beta_start=diffusion_config['beta_start'],
                                     beta_end=diffusion_config['beta_end'])
    with torch.no_grad():
        sample(model, scheduler, train_config, model_config, diffusion_config)


if __name__ == '__main__':
    infer({'config_path': '/mnt/c/Users/Acer/Graduate/Courses/AI 222/mlops-learning/explorations/diffusion/config/default.yaml'})

{'dataset_params': {'im_path': '/mnt/c/Users/Acer/Graduate/Courses/AI 222/mlops-learning/explorations/diffusion/data/train/images'}, 'diffusion_params': {'num_timesteps': 1000, 'beta_start': 0.0001, 'beta_end': 0.02}, 'model_params': {'im_channels': 1, 'im_size': 28, 'down_channels': [32, 64, 128, 256], 'mid_channels': [256, 256, 128], 'down_sample': [True, True, False], 'time_emb_dim': 128, 'num_down_layers': 2, 'num_mid_layers': 2, 'num_up_layers': 2, 'num_heads': 4}, 'train_params': {'task_name': 'default', 'batch_size': 64, 'num_epochs': 40, 'num_samples': 100, 'num_grid_rows': 10, 'lr': 0.0001, 'ckpt_name': '/mnt/c/Users/Acer/Graduate/Courses/AI 222/mlops-learning/explorations/diffusion/ddpm_ckpt.pth'}}


/tmp/ipykernel_224031/3405986119.py:56: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(os.path.join(train_config['task_name'],
1000it [03:45,